# 02 — Baseline Fare Prediction Model

This notebook:
1. Loads the processed dataset from notebook 01
2. Splits into train/test sets
3. Trains an XGBoost regressor as the baseline ("business-as-usual") model
4. Evaluates standard regression metrics
5. Saves model and predictions for the bias audit in notebook 03

**Important:** The sensitive attribute (`majority_race`) is NOT used as a model feature.  
It is carried alongside predictions purely for auditing purposes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
FIG_DIR = '../figures'
os.makedirs(FIG_DIR, exist_ok=True)

FEATURE_COLS = ['trip_miles', 'trip_seconds', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour']
TARGET_COL = 'fare'
SENSITIVE_COL = 'majority_race'

In [ ]:
# ── Load processed data ──────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'trips_processed.csv'))
print(f'Loaded {len(df):,} rows')
print(f'Group distribution:\n{df[SENSITIVE_COL].value_counts()}')

## 2.1 — Train / Test Split

In [ ]:
X = df[FEATURE_COLS]
y = df[TARGET_COL]
sensitive = df[SENSITIVE_COL]

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive,
    test_size=0.2,
    random_state=42
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'\nTest set group distribution:')
print(sens_test.value_counts())

## 2.2 — Train Baseline XGBoost

In [ ]:
model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=50)
print('Training complete.')

## 2.3 — Overall Evaluation

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('=== Baseline Model — Overall Metrics ===')
print(f'  RMSE : {rmse:.3f}')
print(f'  MAE  : {mae:.3f}')
print(f'  R²   : {r2:.4f}')

In [ ]:
# ── Predicted vs Actual scatter ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
sample_idx = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
ax.scatter(y_test.values[sample_idx], y_pred[sample_idx], alpha=0.2, s=8)
lims = [0, y_test.quantile(0.99)]
ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Fare ($)')
ax.set_ylabel('Predicted Fare ($)')
ax.set_title('Baseline Model: Predicted vs Actual')
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'baseline_pred_vs_actual.png'), dpi=150)
plt.show()

## 2.4 — Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax)
ax.set_title('Feature Importance (XGBoost baseline)')
ax.set_xlabel('Importance')
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'feature_importance.png'), dpi=150)
plt.show()

## 2.5 — Quick Per-Group Preview

A brief look at errors by group — the full audit is in notebook 03.

In [ ]:
results = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred,
    'error': y_pred - y_test.values,
    'abs_error': np.abs(y_pred - y_test.values),
    'group': sens_test.values,
})

print('--- Per-group error summary (baseline) ---')
print(results.groupby('group')[['error', 'abs_error']].agg(['mean', 'std']).round(3))

## 2.6 — Save Artifacts

In [ ]:
# Save model
joblib.dump(model, os.path.join(DATA_DIR, 'baseline_model.pkl'))

# Save test set with predictions (for audit notebook)
test_output = X_test.copy()
test_output[TARGET_COL] = y_test.values
test_output['predicted_fare'] = y_pred
test_output[SENSITIVE_COL] = sens_test.values

# Also carry income_group for audit
test_output['income_group'] = df.loc[X_test.index, 'income_group'].values
test_output['price_per_mile'] = df.loc[X_test.index, 'price_per_mile'].values
test_output['pickup_census_tract'] = df.loc[X_test.index, 'pickup_census_tract'].values

test_output.to_csv(os.path.join(DATA_DIR, 'test_with_predictions.csv'), index=False)

# Save train indices for mitigation notebook
train_output = X_train.copy()
train_output[TARGET_COL] = y_train.values
train_output[SENSITIVE_COL] = sens_train.values
train_output.to_csv(os.path.join(DATA_DIR, 'train_set.csv'), index=False)

print('Saved: baseline_model.pkl, test_with_predictions.csv, train_set.csv')